# 🎯 Step 3: Preference Alignment ด้วย DPO 🔴

**ระดับ:** ขั้นสูง | **GPU:** T4 16GB | **เวลา:** ~60 นาที

## DPO คืออะไร?

**DPO (Direct Preference Optimization)** สอนให้โมเดลตอบได้ตรงใจมนุษย์มากขึ้น
โดยเรียนรู้จากคู่คำตอบ **"ดี vs แย่"** โดยตรง ไม่ต้องเทรน Reward Model แยก

```
Dataset DPO:
┌─────────────────────────────────────────────┐
│ prompt:   "อธิบาย recursion ให้หน่อย"       │
│ chosen:   คำตอบที่ดี (มนุษย์เลือก) ✅        │
│ rejected: คำตอบที่แย่ (มนุษย์ไม่ชอบ) ❌      │
└─────────────────────────────────────────────┘

DPO Loss สอนให้โมเดล:
  ✅ เพิ่มความน่าจะเป็นของ chosen
  ❌ ลดความน่าจะเป็นของ rejected
```

## Pipeline ที่แนะนำ
```
Base Model → SFT (Step 1-2) → DPO (Step 3) → Export (Step 4)
```
> ⚠️ ต้องทำ SFT ก่อนเสมอ! DPO ต้องการโมเดลที่รู้จักรูปแบบการตอบแล้ว

---
> ⚡ **เปิด GPU ก่อน!** Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets -q
print('✅ ติดตั้งเสร็จสิ้น')

## 📦 Import Libraries

> ⚠️ `PatchDPOTrainer()` ต้องเรียกก่อนใช้ DPOTrainer เสมอ เพื่อให้ทำงานกับ Unsloth ได้

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth.chat_templates import get_chat_template
from trl import DPOTrainer, DPOConfig
from datasets import Dataset, load_dataset
import torch

# Patch DPOTrainer ให้ทำงานกับ Unsloth ได้ (สำคัญ!)
PatchDPOTrainer()

MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
MAX_SEQ_LENGTH = 2048

## 🚀 โหลดโมเดล

**แนะนำ:** โหลดจาก SFT checkpoint ของ Step 2 เพื่อผลลัพธ์ที่ดีที่สุด

```python
# ถ้าทำ Step 2 แล้ว:
MODEL_NAME = "outputs/lora_model_chat"

# ถ้าไม่มี SFT checkpoint:
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"  # base model โดยตรง
```

In [ ]:
print('📥 กำลังโหลดโมเดล...')

model, tokenizer = FastLanguageModel.from_pretrained(
    # เปลี่ยนเป็น "outputs/lora_model_chat" ถ้าทำ Step 2 แล้ว
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
print('✅ โหลดโมเดลสำเร็จ')

## 🔧 ตั้งค่า LoRA สำหรับ DPO

DPO ใช้ `r` ที่เล็กกว่า SFT ได้ เพราะเราแค่ **"ปรับ"** พฤติกรรม ไม่ได้สอนทักษะใหม่

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,                                                    # เล็กกว่า SFT (16 → 8)
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # เฉพาะ attention
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
model.print_trainable_parameters()

## 📊 เตรียม DPO Dataset

DPO Dataset ต้องมี **3 columns** เสมอ:

| Column | คำอธิบาย |
|--------|----------|
| `prompt` | คำถาม/บริบท |
| `chosen` | คำตอบที่ดี (มนุษย์ prefer) ✅ |
| `rejected` | คำตอบที่แย่ (มนุษย์ไม่ prefer) ❌ |

Dataset จริงที่ใช้ได้:
- `"Anthropic/hh-rlhf"` (English)
- `"Intel/orca_dpo_pairs"` (English)
- `"argilla/ultrafeedback-binarized-preferences"` (English)

In [ ]:
# ── ตัวเลือก A: โหลดจาก HuggingFace ──
# raw_dataset = load_dataset("Intel/orca_dpo_pairs", split="train[:1000]")

# ── ตัวเลือก B: สร้าง Dataset เองสำหรับสาธิต ──
dpo_examples = [
    {
        "prompt": "อธิบาย recursion ใน Python พร้อมตัวอย่าง",
        "chosen": "Recursion คือการที่ฟังก์ชันเรียกตัวเองซ้ำๆ จนกว่าจะถึง base case\n\nตัวอย่าง factorial:\n```python\ndef factorial(n):\n    if n <= 1:  # base case\n        return 1\n    return n * factorial(n - 1)\n\nprint(factorial(5))  # 120\n```\n\nสำคัญ: ต้องมี base case เสมอ ไม่งั้นจะเกิด RecursionError",
        "rejected": "recursion คือฟังก์ชันที่เรียกตัวเอง",
    },
    {
        "prompt": "วิธีจัดการ exception ใน Python",
        "chosen": "ใช้ try-except block:\n\n```python\ntry:\n    result = 10 / 0\nexcept ZeroDivisionError as e:\n    print(f'Error: {e}')\nexcept Exception as e:\n    print(f'Unexpected error: {e}')\nelse:\n    print('สำเร็จ!')\nfinally:\n    print('รันเสมอ')\n```\n\nBest practice: ระบุ exception type ให้ชัดเจน อย่าใช้ bare except",
        "rejected": "ใช้ try except แล้วก็จับ error",
    },
    {
        "prompt": "ความแตกต่างระหว่าง list และ tuple ใน Python",
        "chosen": "ความแตกต่างหลัก:\n\nList []: mutable (แก้ไขได้)\n- my_list = [1, 2, 3]\n- my_list.append(4)  # ได้\n\nTuple (): immutable (แก้ไขไม่ได้)\n- my_tuple = (1, 2, 3)\n- my_tuple[0] = 4  # TypeError!\n\nเมื่อไหร่ใช้อะไร:\n- List: ข้อมูลที่ต้องเปลี่ยนแปลง\n- Tuple: ข้อมูลคงที่ เช่น coordinates, config\n- Tuple เร็วกว่าและประหยัด memory กว่า",
        "rejected": "list ใช้ [] tuple ใช้ () list แก้ได้ tuple แก้ไม่ได้",
    },
    {
        "prompt": "อธิบาย decorator ใน Python",
        "chosen": "Decorator คือ function ที่ wrap function อื่น เพื่อเพิ่มพฤติกรรม\n\n```python\ndef timer(func):\n    import time\n    def wrapper(*args, **kwargs):\n        start = time.time()\n        result = func(*args, **kwargs)\n        print(f'ใช้เวลา: {time.time()-start:.2f}s')\n        return result\n    return wrapper\n\n@timer\ndef my_func():\n    time.sleep(1)\n\nmy_func()  # ใช้เวลา: 1.00s\n```\n\nใช้บ่อยใน: logging, authentication, caching",
        "rejected": "decorator ใช้ @ แล้วก็ wrap function",
    },
]

dataset = Dataset.from_list(dpo_examples)
print(f'✅ DPO Dataset: {len(dataset)} ตัวอย่าง')
print(f'\n📝 ตัวอย่าง:')
print(f'Prompt:   {dataset[0]["prompt"]}')
print(f'Chosen:   {dataset[0]["chosen"][:80]}...')
print(f'Rejected: {dataset[0]["rejected"]}')

## 🔄 Format Dataset เป็น Chat Format

DPOTrainer รองรับ 2 รูปแบบ:
- **String format:** `prompt`, `chosen`, `rejected` เป็น string
- **Chat format:** เป็น list of messages (แนะนำสำหรับ chat model)

In [ ]:
def format_dpo_chat(examples):
    """แปลง string format เป็น chat format สำหรับ DPOTrainer"""
    prompts, chosens, rejecteds = [], [], []
    for prompt, chosen, rejected in zip(
        examples["prompt"], examples["chosen"], examples["rejected"]
    ):
        prompts.append([{"role": "user", "content": prompt}])
        chosens.append([{"role": "assistant", "content": chosen}])
        rejecteds.append([{"role": "assistant", "content": rejected}])
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

dataset = dataset.map(format_dpo_chat, batched=True)
print('✅ Format เสร็จสิ้น')

## 🏋️ ตั้งค่าและรัน DPOTrainer

| พารามิเตอร์ | ค่า | คำอธิบาย |
|------------|-----|----------|
| `beta` | 0.1 | DPO temperature — ยิ่งสูง โมเดลเปลี่ยนแปลงน้อยลง (แนะนำ 0.1-0.5) |
| `max_prompt_length` | 512 | ความยาวสูงสุดของ prompt |
| `max_length` | 1024 | ความยาวสูงสุดรวม (prompt + response) |
| `ref_model` | None | None = ใช้ copy ของ model เป็น reference (ประหยัด VRAM) |

> ⚠️ DPO ต้องการ VRAM มากกว่า SFT เพราะโหลด 2 โมเดล (policy + reference)

In [ ]:
dpo_config = DPOConfig(
    # ── DPO Specific ──
    beta=0.1,                    # DPO temperature (ค่าแนะนำ: 0.1-0.5)
    max_prompt_length=512,
    max_length=1024,

    # ── Batch & Gradient ──
    per_device_train_batch_size=1,   # DPO ต้องการ VRAM มากกว่า ใช้ 1
    gradient_accumulation_steps=8,

    # ── Learning Rate ──
    warmup_ratio=0.1,
    num_train_epochs=1,
    learning_rate=5e-5,              # DPO ใช้ lr ต่ำกว่า SFT
    lr_scheduler_type="cosine",

    # ── Precision ──
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    optim="adamw_8bit",
    logging_steps=5,
    output_dir="outputs",
    save_strategy="epoch",
    seed=42,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,      # None = ใช้ copy ของ model เป็น reference (ประหยัด VRAM)
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('🏋️  เริ่มการเทรนด้วย DPO...')
print('⚠️  ถ้า OOM ให้ลด max_length หรือ max_prompt_length')
trainer_stats = trainer.train()
print(f'\n✅ DPO เทรนเสร็จสิ้น!')
print(f'⏱️  เวลาที่ใช้: {trainer_stats.metrics["train_runtime"]:.2f} วินาที')

## 💾 บันทึกโมเดล

In [ ]:
model.save_pretrained("outputs/lora_model_dpo")
tokenizer.save_pretrained("outputs/lora_model_dpo")
print('✅ บันทึกสำเร็จที่ outputs/lora_model_dpo/')

## 🧪 ทดสอบโมเดลหลัง DPO

In [ ]:
FastLanguageModel.for_inference(model)

# 🔧 เปลี่ยนคำถามได้ที่นี่
test_messages = [
    {"role": "user", "content": "อธิบาย recursion ใน Python พร้อมตัวอย่าง"},
]

input_text = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True,
)

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs, max_new_tokens=300, temperature=0.7, do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print('📤 คำตอบจากโมเดลหลัง DPO:')
print('─' * 50)
print(response)
print('─' * 50)
print('\n🎉 Step 3 เสร็จสมบูรณ์! ไปต่อที่ Step 4 (Export) ได้เลย')